In [68]:
import pandas as pd

file_path = "/Users/huahua/Desktop/train.csv"

# Read the CSV file
df = pd.read_csv(file_path, delimiter=";")

# Display first few rows
print(df.head())


   age           job  marital  education default  balance housing loan  \
0   58    management  married   tertiary      no     2143     yes   no   
1   44    technician   single  secondary      no       29     yes   no   
2   33  entrepreneur  married  secondary      no        2     yes  yes   
3   47   blue-collar  married    unknown      no     1506     yes   no   
4   33       unknown   single    unknown      no        1      no   no   

   contact  day month  duration  campaign  pdays  previous poutcome   y  
0  unknown    5   may       261         1     -1         0  unknown  no  
1  unknown    5   may       151         1     -1         0  unknown  no  
2  unknown    5   may        76         1     -1         0  unknown  no  
3  unknown    5   may        92         1     -1         0  unknown  no  
4  unknown    5   may       198         1     -1         0  unknown  no  


# generate conversion rate column

In [69]:
# Convert 'y' to binary (1 for 'yes', 0 for 'no')
df['conversion_binary'] = df['y'].apply(lambda x: 1 if x == 'yes' else 0)

# Calculate conversion rate (success rate per contact attempt)
df['conversion_rate'] = df['conversion_binary'] / df['campaign']

# Display the first few rows
print(df[['y', 'campaign', 'conversion_rate']].head())

    y  campaign  conversion_rate
0  no         1              0.0
1  no         1              0.0
2  no         1              0.0
3  no         1              0.0
4  no         1              0.0


In [70]:
# Sort dataframe by conversion_rate in descending order
df_sorted = df.sort_values(by='conversion_rate', ascending=False)

# Display the first few rows after sorting
print(df_sorted[['y', 'campaign', 'conversion_rate']].head())


         y  campaign  conversion_rate
34050  yes         1              1.0
8731   yes         1              1.0
36792  yes         1              1.0
42835  yes         1              1.0
42834  yes         1              1.0


# generate best time to contact col

In [71]:
contact_time_mapping = {
    "student": "6-8pm",
    "retired": "12-2pm",
    "unemployed": "12-2pm",
    "housemaid": "2-4pm",
    "admin.": "4-5pm",
    "management": "4-5pm",
    "entrepreneur": "4-5pm",
    "blue-collar": "4-5pm",
    "self-employed": "4-5pm",
    "technician": "4-5pm",
    "services": "4-5pm",
    "unknown": "4-5pm"
}

# Apply mapping to create the new column
df["best_contact_time"] = df["job"].map(contact_time_mapping)

# Display result
print(df[["job", "best_contact_time"]].head())

            job best_contact_time
0    management             4-5pm
1    technician             4-5pm
2  entrepreneur             4-5pm
3   blue-collar             4-5pm
4       unknown             4-5pm


# generate fatigue score

In [72]:
# Define decay factor (adjust as needed)
df["decay_factor"] = df.apply(lambda row: 0.7 if (row["campaign"] + row["previous"]) > 5 else 0.5, axis=1)

# Calculate Fatigue Score
df["fatigue_score"] = (df["campaign"] + df["previous"]) * df["decay_factor"]

# Display the results
print(df[["campaign", "previous", "decay_factor", "fatigue_score"]].head())


   campaign  previous  decay_factor  fatigue_score
0         1         0           0.5            0.5
1         1         0           0.5            0.5
2         1         0           0.5            0.5
3         1         0           0.5            0.5
4         1         0           0.5            0.5


In [73]:
# Ensure conversion column is binary
df['conversion_binary'] = df['y'].apply(lambda x: 1 if x == 'yes' else 0)

# Create age groups
df['age_group'] = pd.cut(df['age'], bins=[18, 30, 40, 50, 60, 100], labels=["18-30", "30-40", "40-50", "50-60", "60+"])

# Income-based segmentation using balance
df['income_group'] = pd.cut(df['balance'], bins=[-2000, 0, 5000, 20000, 50000, 100000], 
                             labels=["Debt", "Low", "Middle", "High", "Very High"])

# Engagement groups based on previous campaign responses
df['engagement_group'] = df['poutcome'].map({"success": "high", "failure": "low", "other": "medium", "unknown": "new"})

# Group people based on key characteristics
grouped_df = df.groupby(['age_group', 'job', 'marital', 'income_group', 'engagement_group'])

# Check how many groups we have
print(f"Total groups: {len(grouped_df)}")


Total groups: 1235


/var/folders/3j/51w1rjp53fd_5rt5m6xczh4w0000gn/T/ipykernel_93579/3578204287.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_df = df.groupby(['age_group', 'job', 'marital', 'income_group', 'engagement_group'])


In [74]:
# Calculate success rate per group
best_strategy_df = grouped_df.agg({
    'conversion_binary': 'mean',  # Conversion rate
    'contact': lambda x: x.value_counts().idxmax(),  # Most successful contact method
    'best_contact_time': lambda x: x.value_counts().idxmax(),  # Most successful contact time
    'previous': 'median',  # Typical number of past contacts
}).reset_index()

# Rename for clarity
best_strategy_df.rename(columns={'conversion_binary': 'conversion_rate'}, inplace=True)

# Display best strategies
print(best_strategy_df.head())


  age_group     job   marital income_group engagement_group  conversion_rate  \
0     18-30  admin.  divorced         Debt             high              NaN   
1     18-30  admin.  divorced         Debt              low              NaN   
2     18-30  admin.  divorced         Debt           medium              NaN   
3     18-30  admin.  divorced         Debt              new            0.375   
4     18-30  admin.  divorced          Low             high            1.000   

    contact best_contact_time  previous  
0       NaN               NaN       NaN  
1       NaN               NaN       NaN  
2       NaN               NaN       NaN  
3   unknown             4-5pm       0.0  
4  cellular             4-5pm       1.0  


In [75]:
from scipy.stats import beta
from collections import defaultdict

# Initialize reward tracking per group
group_strategy_rewards = defaultdict(lambda: defaultdict(lambda: {'success': 0, 'failures': 0}))

# Loop through dataset and track successes/failures for each group & strategy
for _, row in df.iterrows():
    group = (row['age_group'], row['job'], row['marital'], row['income_group'], row['engagement_group'])
    strategy = (row['best_contact_time'], row['contact'])  # Contact time + method
    conversion = row['conversion_binary']
    
    if conversion == 1:
        group_strategy_rewards[group][strategy]['success'] += 1
    else:
        group_strategy_rewards[group][strategy]['failures'] += 1

# Function to pick the best strategy per group
def select_best_strategy_per_group(group_strategy_rewards):
    best_strategies = {}

    for group, strategies in group_strategy_rewards.items():
        best_strategy = None
        best_beta_sample = -1
        
        for strategy, rewards in strategies.items():
            sampled_beta = beta.rvs(rewards['success'] + 1, rewards['failures'] + 1)
            if sampled_beta > best_beta_sample:
                best_beta_sample = sampled_beta
                best_strategy = strategy
        
        best_strategies[group] = best_strategy

    return best_strategies

# Get the best strategy for each group
optimal_strategies = select_best_strategy_per_group(group_strategy_rewards)

# Display best strategies per customer segment
for group, strategy in optimal_strategies.items():
    print(f"Customer Segment: {group} --> Best Contact Strategy: {strategy}")


Customer Segment: ('50-60', 'management', 'married', 'Low', 'new') --> Best Contact Strategy: ('4-5pm', 'cellular')
Customer Segment: ('40-50', 'technician', 'single', 'Low', 'new') --> Best Contact Strategy: ('4-5pm', 'cellular')
Customer Segment: ('30-40', 'entrepreneur', 'married', 'Low', 'new') --> Best Contact Strategy: ('4-5pm', 'telephone')
Customer Segment: ('40-50', 'blue-collar', 'married', 'Low', 'new') --> Best Contact Strategy: ('4-5pm', 'cellular')
Customer Segment: ('30-40', 'unknown', 'single', 'Low', 'new') --> Best Contact Strategy: ('4-5pm', 'cellular')
Customer Segment: ('30-40', 'management', 'married', 'Low', 'new') --> Best Contact Strategy: ('4-5pm', 'telephone')
Customer Segment: ('18-30', 'management', 'single', 'Low', 'new') --> Best Contact Strategy: ('4-5pm', 'telephone')
Customer Segment: ('40-50', 'entrepreneur', 'divorced', 'Low', 'new') --> Best Contact Strategy: ('4-5pm', 'telephone')
Customer Segment: ('50-60', 'retired', 'married', 'Low', 'new') --> 